# TerraScope Scientific Validation Standards & Cross-Validation

This notebook demonstrates and executes the three core validation standards defined in **Section 4 of Data Sources & Ingestion Standards**:

1. **NASA Giovanni**: Monthly time series extraction for ground-truth comparison (MERRA-2 `M2TMNXFLX` and GPM IMERG `GPM_3IMERGM`).
2. **Data Rods for Hydrology**: Point time series from GLDAS (`GLDAS_NOAH025_M_2_1`) for soil moisture verification.
3. **KNMI Climate Explorer**: Cross-validation of Mann-Kendall statistics ($S$, $Z$, $p$-value, Sen's slope).


In [ ]:
import json
import math
from pathlib import Path
import sys

# Ensure project root is in sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from backend.services.validation import (
    GiovanniValidationService,
    DataRodsHydrologyValidationService,
    KnmiClimateExplorerValidationService
)
from backend.services.nasa_power import NasaPowerService
from backend.services.swot import SwotService
from backend.services.gldas import GldasService

print("TerraScope Validation Framework loaded successfully.")

## 1. NASA Giovanni Ground-Truth Comparison
- Compares NASA POWER aggregated series with NASA GES DISC Giovanni area-averaged monthly series.
- Evaluates Root-Mean-Square Difference (RMSD) and Pearson correlation ($r$).

In [ ]:
sample_file = project_root / "data" / "sample" / "kerala_temperature_sample.json"
with open(sample_file, "r", encoding="utf-8") as f:
    sample_data = json.load(f)

power_series = [pt["value"] for pt in sample_data["results"][0]["series"]]
# Simulated reference Giovanni MERRA-2 series closely tracking ground truth
giovanni_series = [round(v + 0.12 * math.sin(i), 2) for i, v in enumerate(power_series)]

validator = GiovanniValidationService()
comparison = validator.compare_series("temperature", power_series, giovanni_series, tolerance_rmsd=0.5)
print("Giovanni Comparison Result:")
print(json.dumps(comparison, indent=2))

## 2. Data Rods for Hydrology (GLDAS Soil Moisture Verification)
- Validates NASA POWER surface soil wetness (`GWETTOP`) against GLDAS point time series.

In [ ]:
gldas_service = GldasService()
print("GLDAS Proxy Metadata:")
print(json.dumps(gldas_service.get_proxy_indicator_metadata(), indent=2))

data_rods_val = DataRodsHydrologyValidationService()
# Synthetic test verification series
power_soil = [0.45, 0.48, 0.52, 0.50, 0.43, 0.47, 0.55]
gldas_soil = [0.43, 0.46, 0.50, 0.49, 0.41, 0.45, 0.53]
soil_verif = data_rods_val.verify_soil_moisture(power_soil, gldas_soil)
print("\nSoil Moisture Verification Result:")
print(json.dumps(soil_verif, indent=2))

## 3. KNMI Climate Explorer Cross-Validation
- Cross-validates computed Mann-Kendall statistics ($p$-value, Sen's slope) against KNMI Climate Explorer benchmarks.

In [ ]:
val_standards_file = project_root / "data" / "sample" / "validation_standards.json"
with open(val_standards_file, "r", encoding="utf-8") as f:
    standards_doc = json.load(f)

knmi_benchmark = standards_doc["standards"]["knmi_climate_explorer"]["ground_truth_statistics"]
computed_stats = sample_data["results"][0]["trend"]

knmi_validator = KnmiClimateExplorerValidationService()
knmi_result = knmi_validator.validate_trend_statistics(computed_stats, knmi_benchmark)
print("KNMI Cross-Validation Result:")
print(json.dumps(knmi_result, indent=2))